# Partitioning Fundamentals: Parallelism and Data Distribution

**Module**: 05_partitions_and_shuffle  
**Notebook**: 01_partitioning_fundamentals.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Spark partitions datasets, the correlation between partitions and executor tasks, and inspecting row distributions.
- **Why does this topic exist?**: Partitions dictate cluster concurrency, memory footprint, and network transfer volume.
- **What problem does it solve?**: Improper partition sizing causes either underutilized hardware or out-of-memory errors.
- **Where is it used in real-world Data Engineering?**: Sizing data lake ingestion and tuning shuffle partitions.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | RDD partitions, lazy evaluation, cluster architecture |
| **Required depth** | Working knowledge: can explain parallelism, partition sizing, and the cost of too many/too few partitions |
| **Data Engineer** | Choose partition counts that keep tasks busy without creating tiny-task overhead. |
| **Senior Data Engineer** | Diagnose parallelism problems from the UI and relate them to input splits and shuffle partitions. |
| **Spark specialist** | Reason about partition sizing, task scheduling, and file/block boundaries. |
| **Next topic** | `05_partitions_and_shuffle/02_repartition_and_coalesce.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

A partition is the unit of parallel work in Spark. Too few partitions leave cores idle; too many create
scheduling overhead and tiny tasks. Partition size also determines how much memory each task needs and how evenly
the cluster is utilized. Partitioning is therefore the first performance knob, not an afterthought.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Partition** | A slice of data processed by one task | The unit of parallelism for RDDs and DataFrames |
| **Parallelism** | Number of concurrent tasks | Bounded by partitions and available cores |
| **Input split** | How a file source divides data | Often one partition per file block/split |
| **Shuffle partition** | Output partition of a wide operation | Controlled by `spark.sql.shuffle.partitions` (default 200) |
| **Partition size** | Bytes/rows per partition | A heuristic target, not a universal constant |
| **Task overhead** | Cost per task | Scheduling, serialization, and startup add up with many tiny tasks |

### How it works, step by step

1. The input source creates partitions based on files, blocks, or splits.
2. The scheduler creates one task per partition for each stage.
3. Tasks are assigned to executor cores; extra tasks wait in the queue.
4. Each task processes its partition and emits records to the next stage.
5. A shuffle repartitions records according to the target partition count and key.
6. The next stage creates one task per shuffle partition.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Partition Sizing Heuristics

- **Target Size**: 100 MB to 200 MB in memory per partition.
- **Cluster Ratio**: 2x to 4x total executor cores in the cluster.
- Our Docker Cluster: 2 workers x 2 cores = 4 cores -> Target 8 to 16 partitions for active processing.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Partitions_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Inspecting File Partitions on Retail Orders (60,000 rows)

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
print(f"Total orders: {orders.count()}")
print(f"Initial partition count from disk: {orders.rdd.getNumPartitions()}")

# Inspect row distribution per partition using glom
dist = orders.rdd.glom().map(len).collect()
print(f"Rows per partition: {dist}")

---

## Experiment: Parallelism Scaling with Partition Count

- **1. What are we testing?**: Compare execution time of heavy CPU map across 1, 2, and 4 partitions.
- **2. What do we expect?**: Execution time scales inversely with partition count up to 4 cores.

In [ ]:
def cpu_heavy(row):
    return sum(i*i for i in range(1000))

for p in [1, 2, 4]:
    t0 = time.time()
    orders.repartition(p).rdd.map(cpu_heavy).count()
    print(f"Partitions {p}: {time.time()-t0:.3f}s")

**3. What happened?**: Increasing partitions up to total cores decreased runtime.  
**4. Why did it happen?**: More cores were actively utilized in parallel.  
**5. Practical Engineering Lesson**: Always maintain at least 1 partition per available core.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: IoT sensor backfill

- **Industry**: Industrial IoT
- **Scenario**: A backfill reads years of sensor files and runs a per-device aggregation.
- **Problem**: The input has thousands of tiny files, so Spark creates thousands of tiny tasks and spends more time scheduling than computing.
- **Why this concept is relevant**: Compacting files and choosing a sensible partition count reduces task overhead and improves throughput.
- **Implementation idea**: Compare task count, median task duration, and total runtime before and after compaction.
- **Expected result**: Fewer, larger tasks and a shorter backfill.
- **Alternative**: Increase parallelism to hide the overhead.
- **Trade-offs**: More parallelism cannot fix per-task overhead when each task is only milliseconds long.

### Use Case 2: Fraud scoring at peak

- **Industry**: Financial services
- **Scenario**: A scoring job uses the default 200 shuffle partitions on a small cluster with 4 cores.
- **Problem**: Most partitions are tiny, and the job spends time scheduling 200 tasks for a small dataset.
- **Why this concept is relevant**: Sizing shuffle partitions to the data and cluster reduces overhead while preserving enough parallelism.
- **Implementation idea**: Measure shuffle partition sizes and task duration distribution, then adjust the partition count.
- **Expected result**: A stage with tasks large enough to amortize overhead and enough tasks to use all cores.
- **Alternative**: Enable AQE coalescing.
- **Trade-offs**: AQE can merge small partitions automatically, but an explicit count still helps when statistics are stable.

---

## Business Value

- Better cluster utilization and lower job cost.
- More predictable runtimes for recurring pipelines.
- Fewer small-file and tiny-task problems.
- A shared sizing method for data and platform teams.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Aim for partitions large enough to amortize task overhead, but small enough to fit comfortably in memory.
- Do not hard-code a universal partition size; measure your own workload.
- Account for input splits, shuffle partitions, and output files separately.
- Check the task-duration distribution, not just the average.
- Let AQE coalesce shuffle partitions when the data size varies.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Find the parallelism knee

**Steps**

1. Run the same aggregation with 2, 8, 32, and 128 shuffle partitions on the lab cluster.
2. Record runtime, task count, median task duration, and shuffle bytes.
3. Identify where adding partitions stops helping.

**What to observe**: Runtime improves until there are enough tasks to use the cores, then flattens or worsens as tasks become tiny.

**Healthy result**: The chosen count uses the available cores without excessive task overhead.

**Problematic result**: Hundreds of tasks finish in milliseconds while a few large tasks dominate.

**Engineering takeaway**: Partition count should follow data size and cluster capacity, not a copied default.

---

## Trade-offs and Alternatives

### Advantages

- Direct control of parallelism
- Simple mental model
- Works with all APIs

### Limitations

- Optimal size is workload-dependent
- Too many partitions add overhead
- Too few leave cores idle

### When to use it

- Initial job sizing
- Backfills
- Shuffle tuning

### When not to use it

- Copying a partition count blindly
- Ignoring task duration
- Partitioning by unique IDs

### Alternatives inside Spark

- repartition
- coalesce
- AQE
- Input splits

### Alternatives outside Spark

- File compaction
- Table layout design
- Cluster autoscaling

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Default Shuffle Partitions
**Problem Statement**: Retrieve spark.sql.shuffle.partitions.  
**Input / Context**: spark.conf

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Custom Repartition and Glom
**Problem Statement**: Repartition orders into 6 partitions and print min/max partition sizes.  
**Input / Context**: orders

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Partition Count Calculation
**Problem Statement**: Write a Python function to compute optimal partition count given dataset size in MB and core count.  
**Input / Context**: Formula

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The 200-partition small job

**Scenario**: A 2 GB aggregation runs with 200 shuffle partitions on a four-core cluster and takes longer than expected.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Inspect shuffle partition sizes and task durations.
2. Check whether AQE is enabled and coalescing.
3. Try a smaller partition count and compare total runtime.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Default Shuffle Partitions):
```python
print(spark.conf.get('spark.sql.shuffle.partitions'))
```

#### Solution 2 (Custom Repartition and Glom):
```python
s = orders.repartition(6).rdd.glom().map(len).collect()
print('Sizes:', s,'Min:', min(s),'Max:', max(s))
```

#### Solution 3 (Partition Count Calculation):
```python
def calc_partitions(size_mb, cores): return max(cores * 2, int(size_mb / 128))
print('Optimal for 10GB on 16 cores:', calc_partitions(10240, 16))
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Sizing Rule**: 100-200 MB per partition; 2-4x total cores.

### Essential Technical Terms
- `Partition`: Atomic unit of distributed data.
- `glom`: Inspects partition contents.

### Common Engineering Mistakes
- **Single partition**: Serializes execution on 1 core.

### Interview Questions on This Topic
- **Q: How does partition count affect task count?**
  - *A*: 1:1 correlation per stage.


### More Interview Questions

- **Q: What determines the number of tasks in a stage?**
  - *A*: The number of partitions in that stage; each partition becomes one task, subject to available executor cores.
- **Q: Why can too many partitions be slower?**
  - *A*: Each task incurs scheduling, serialization, and startup overhead; with tiny partitions, overhead can dominate the actual work.

### What to Study Next
Open 02_repartition_and_coalesce.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')